In [1]:
import os
os.chdir('D:\\ai-engineering-buildcamp-codespace\\Homework\\')
print(os.getcwd())

D:\ai-engineering-buildcamp-codespace\Homework


In [42]:
import pandas as pd
import requests
from pathlib import Path
from urllib.parse import urlparse
from markitdown import MarkItDown
from typing import Any, Dict, Iterable, List
import json
from pydantic import BaseModel, Field
from typing import Literal
from pydantic_ai import Agent
from gitsource import chunk_documents
from typing import Any, Dict, List
from minsearch import AppendableIndex, Highlighter, Tokenizer
from minsearch.tokenizer import DEFAULT_ENGLISH_STOP_WORDS
from typing import List, Dict, Callable
import inspect
from pydantic_ai import Agent, RunContext

In [3]:
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())

True

In [4]:
from openai import OpenAI
openai_client  = OpenAI()

Wikipedia API Setup


We'll use two Wikipedia API endpoints.


Search API - Find pages related to a topic:


https://en.wikipedia.org/w/api.php?action=query&format=json&list=search&srsearch=YOUR_QUERY


Replace YOUR_QUERY with your search term. Use + for spaces (for example, lesser+capybara).


Get Page API - Fetch the raw content of a page:

https://en.wikipedia.org/w/index.php?title=PAGE_TITLE&action=raw

Replace PAGE_TITLE with the exact title of the Wikipedia page.

Wikipedia requires a User-Agent header on API requests. Without it, your requests may be blocked. Pass it in every requests.get call:

import requests

headers = {"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"}
response = requests.get(url, headers=headers)

You can test these APIs directly in your browser to see how they work.

In [ ]:


def search_wikipedia(query: str) -> list[dict]:
    """Search Wikipedia for pages related to a query."""
    
    # The base URL for Wikipedia API
    url = "https://en.wikipedia.org/w/api.php"
    
    # Required header
    headers = {
        "User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"
    }
    
    # Parameters explain WHAT we want and HOW
    params = {
        "action": "query",      # WHAT: I want to query/retrieve data
        "format": "json",       # HOW: Send response as JSON
        "list": "search",       # WHAT: Give me a list of search results
        "srsearch": query       # WHAT: Search for this term
    }
    
    # Make the request
    # requests.get() will convert params dict into URL query string
    response = requests.get(url, headers=headers, params=params)
    
    # Parse the JSON response
    data = response.json()
    
    # Extract the search results from the nested structure
    # data["query"]["search"] contains the list of results
    search_results = data.get("query", {}).get("search", [])
    
    return search_results


# Function 2: Get Page Content
def get_page(page_title: str) -> str:
    """Fetch the raw content of a Wikipedia page."""
    url = "https://en.wikipedia.org/w/index.php"
    headers = {"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"}
    params = {
        "title": page_title,
        "action": "raw"
    }
    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()
    return response.text




In [ ]:
query = "capybara"

In [23]:
# Test it
search_results = search_wikipedia(query)
print(f"Found {len(results)} results")

Found 10 results


In [25]:
count = 0
for result in search_results:
    if query in result['title'].lower():
        count += 1
        print(f"✓ {result['title']}")
    else:
        print(f"✗ {result['title']}")

print(f"\nAnswer: {count}")

✓ Capybara
✓ Lesser capybara
✓ Capybara (disambiguation)
✓ Capybara Games
✗ Hydrochoerus
✗ Caviidae
✓ Capybara (software)
✗ Flow (2024 film)
✗ Yuzu bath
✗ List of largest rodents

Answer: 5


Question 3. Define the Get Page Tool


Create a get_page function that:

Takes a page title as input

Calls the Wikipedia Get Page API

Returns the page content as a string

Test your function by fetching the page titled "Capybara".

In [31]:
# Step 2: Collect documents from multiple pages
documents = []

for result in search_results:
    page_title = result['title']
    
    # Skip if "capybara" not in title (optional filter)
    if query in page_title.lower():
        
        page_content = get_page(page_title)
        
        # Add to documents list
        documents.append({
            "title": page_title,
            "content": page_content,
            "filename": f"{page_title}.txt",
            "description": result.get('snippet', '')
        })
        
        print(f"  ✓ Got {len(page_content):,} characters\n")
        


# Step 3: Chunk ALL documents at once
print(f"Chunking {len(documents)} documents...")
chunked_docs = chunk_documents(
    documents,
    size=3000,
    step=1500,
    content_field_name="content"
)

print(f"Created {len(chunked_docs)} total chunks\n")

  ✓ Got 36,946 characters

  ✓ Got 5,578 characters

  ✓ Got 516 characters

  ✓ Got 10,540 characters

  ✓ Got 7,859 characters

Chunking 5 documents...
Created 40 total chunks



Question 4. Agent Setup

Now create an agent with access to both tools:

search_wikipedia - Search for Wikipedia pages

get_page - Fetch the content of a specific page

 

In [37]:
# Helper function to auto-extract methods
def get_instance_methods(obj) -> List[Callable]:
    """Extract all public methods from an instance."""
    methods = []
    for name, method in inspect.getmembers(obj, predicate=inspect.ismethod):
        if not name.startswith('_'):
            methods.append(method)
    return methods

In [38]:
class SearchTools:
    """Provides search and retrieval of Wikipedia pages."""
    
    def search_wikipedia(self, query: str) -> List[Dict]:
        """Search Wikipedia for pages related to a query."""
        url = "https://en.wikipedia.org/w/api.php"
        headers = {"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"}
        params = {
            "action": "query",
            "format": "json",
            "list": "search",
            "srsearch": query
        }
        response = requests.get(url, headers=headers, params=params)
        data = response.json()
        return data.get("query", {}).get("search", [])
    
    def get_page(self, page_title: str) -> str:
        """Fetch the raw content of a Wikipedia page."""
        url = "https://en.wikipedia.org/w/index.php"
        headers = {"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"}
        params = {
            "title": page_title,
            "action": "raw"
        }
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()
        return response.text

In [40]:
# Instructions
instructions = """
You are a helpful Wikipedia assistant.

You have access to two tools:
1. search_wikipedia(query: str) - Search Wikipedia for pages related to a query
2. get_page(page_title: str) - Fetch the full content of a specific Wikipedia page

When a user asks about a Wikipedia page URL (e.g., https://en.wikipedia.org/wiki/Capybara):
- Extract the page title from the URL (e.g., "Capybara")
- Use get_page() to fetch the content
- Provide a concise, informative summary

When a user asks a general question:
- Use search_wikipedia() to find relevant pages
- Use get_page() to get detailed content from the most relevant page
- Provide an accurate answer based on the content

Always be concise and accurate.
"""


In [43]:
# Initialize and extract tools
search_tools = SearchTools()
tools = get_instance_methods(search_tools)

In [44]:
# Create agent
search_agent = Agent(
    name='search',
    model='openai:gpt-4o-mini',
    instructions=instructions,
    tools=tools
)

In [46]:
question = "What is this page about? https://en.wikipedia.org/wiki/Capybara"
result = await search_agent.run(question)

In [49]:
question = "What is this page about? https://en.wikipedia.org/wiki/Capybara"
result = await search_agent.run(question)

print("Question:", question)
print("\nAgent's Answer:")
print(result.output)  # ← Use this!

Question: What is this page about? https://en.wikipedia.org/wiki/Capybara

Agent's Answer:
The Wikipedia page for the **capybara** (scientific name: *Hydrochoerus hydrochaeris*) details this animal as the largest living rodent, native to South America, except for Chile. Capybaras are semi-aquatic herbivores, primarily inhabiting savannas and dense forests near water bodies, where they mainly feed on grasses and aquatic plants. 

Capybaras are known for their social behavior, often living in groups ranging from 10 to 20 individuals, though larger gatherings can occur. They are hunted for their meat and hide and have a stable population across much of their range, although hunting in some areas poses a risk.

The page covers various topics such as their physical description, ecology, dietary habits, social structure, reproduction, communication, and conservation status. Capybaras have gained cultural significance and a fan following in various parts of the world, especially in Japan and 

In [50]:
# Ask the question
question = "What are the main threats to capybara populations?"

result = await search_agent.run(question)

print("Question:", question)
print("\nAgent's Answer:")
print(result.output)

print("\n" + "="*60)
print("TOOL CALLS ANALYSIS")
print("="*60)

Question: What are the main threats to capybara populations?

Agent's Answer:
The primary threats to capybara populations largely stem from human activities and environmental changes. Here are the main threats identified:

1. **Habitat Destruction**: Urbanization and agricultural expansion reduce the natural habitats available for capybaras, which rely on wetlands and grasslands.

2. **Hunting and Poaching**: Capybaras are hunted for their meat and skin, which poses a direct threat to their populations.

3. **Agricultural Practices**: Intensive farming practices can lead to habitat degradation. The use of pesticides and fertilizers can also pollute water sources, impacting capybaras and their food supply.

4. **Climate Change**: Changes in climate patterns can disrupt the availability of water and food resources essential for capybaras, particularly in their natural habitats like the Pantanal wetlands.

5. **Infrastructural Development**: Construction of roads and urban areas can fragm

In [51]:
# Get all messages
messages = result.all_messages()

In [52]:
# Count tool calls
tool_call_count = 0

In [53]:
# Analyze messages
for m in messages:
    print(m.kind)
    for p in m.parts:
        part_kind = p.part_kind
        if part_kind == 'user-prompt':
            print('USER:', p.content)
        if part_kind == 'tool-call':
            print('TOOL CALL:', p.tool_name, p.args)
            tool_call_count += 1
        if part_kind == 'tool-return':
            print('TOOL RETURN:', p.tool_name)
        if part_kind == 'text':
            print(p.content)
    print()

request
USER: What are the main threats to capybara populations?

response
TOOL CALL: search_wikipedia {"query":"Capybara threats"}

request
TOOL RETURN: search_wikipedia

response
TOOL CALL: get_page {"page_title":"Pantanal"}

request
TOOL RETURN: get_page

response
The primary threats to capybara populations largely stem from human activities and environmental changes. Here are the main threats identified:

1. **Habitat Destruction**: Urbanization and agricultural expansion reduce the natural habitats available for capybaras, which rely on wetlands and grasslands.

2. **Hunting and Poaching**: Capybaras are hunted for their meat and skin, which poses a direct threat to their populations.

3. **Agricultural Practices**: Intensive farming practices can lead to habitat degradation. The use of pesticides and fertilizers can also pollute water sources, impacting capybaras and their food supply.

4. **Climate Change**: Changes in climate patterns can disrupt the availability of water and f

In [54]:
print("="*60)
print(f"TOTAL TOOL CALLS: {tool_call_count}")
print("="*60)

TOTAL TOOL CALLS: 2
